
# BPY518 Lecture 2: Image Preprocessing and Noise Reduction
Instructor: Josh Shaevitz  
Email: shaevitz@princeton.edu



## Lecture Outline
- Bit depth, dynamic range, and contrast
- Types of noise
- Noise filtering techniques
- Background subtraction



We will consider **single-channel grayscale images** in this lecture. The main goal is to understand how raw microscopy images get distorted by digitization, noise, and background, and what simple preprocessing steps are useful before downstream analysis.


In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'



## Image Bit Depth

Light intensities are digitized into integers. If an image uses `N` bits, it can represent `2**N` distinct gray levels.

- 8-bit: 0 to 255
- 12-bit: 0 to 4095
- 16-bit: 0 to 65,535

The original slide deck compared the same image at four different bit depths. Here is that comparison rebuilt from the PowerPoint media assets.

![Bit depth comparison](media/Lecture_2/bit_depth_comparison.png)


In [ ]:

# Quantize the same smooth grayscale ramp to several bit depths.
gradient = np.tile(np.linspace(0, 1, 256), (80, 1))


def quantize_to_n_bits(image, n_bits):
    levels = 2**n_bits - 1
    return np.round(image * levels) / levels


fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, n_bits in zip(axes, [2, 4, 8, 16]):
    q = quantize_to_n_bits(gradient, n_bits)
    ax.imshow(q, vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'{n_bits}-bit')
    ax.axis('off')

plt.tight_layout()
plt.show()



## Dynamic Range and Contrast

Dynamic range describes the span between the darkest and brightest intensities an image can represent. A common engineering definition is

$$
DR = 20 \log_{10}\left(\frac{I_{\max}}{I_{\min}}\right).
$$

Contrast is about how well the image uses that range.

- A high-contrast image has bright whites and dark blacks with strong edge detail.
- A low-contrast image looks flat because the pixel values occupy only a narrow range.
- High dynamic range and low contrast can happen together when the available range is large but the measured values cluster tightly.
- Stretching contrast improves visibility, but it does **not** create new information.


In [ ]:

# Simulate a low-contrast image and then stretch it.
low_contrast = 0.45 + 0.12 * gradient
stretched = (low_contrast - low_contrast.min()) / (low_contrast.max() - low_contrast.min())

fig, axes = plt.subplots(2, 2, figsize=(10, 5))
axes[0, 0].imshow(low_contrast, vmin=0, vmax=1, aspect='auto')
axes[0, 0].set_title('Low-contrast image')
axes[0, 0].axis('off')

axes[0, 1].imshow(stretched, vmin=0, vmax=1, aspect='auto')
axes[0, 1].set_title('After contrast stretching')
axes[0, 1].axis('off')

axes[1, 0].hist((255 * low_contrast).ravel(), bins=32, color='dimgray')
axes[1, 0].set_title('Original histogram')
axes[1, 0].set_xlabel('Pixel value')
axes[1, 0].set_ylabel('Count')

axes[1, 1].hist((255 * stretched).ravel(), bins=32, color='steelblue')
axes[1, 1].set_title('Stretched histogram')
axes[1, 1].set_xlabel('Pixel value')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()



## Noisy Images

The slide deck distinguishes several major noise sources:

- **Shot noise**: stochastic fluctuations from a finite number of photons.
- **Dark current**: detector signal even without incoming photons.
- **Read noise**: electronic noise from amplification and digitization.

The original PowerPoint contains this comparison figure directly.

![Noise comparison](media/Lecture_2/noise_comparison.png)


In [ ]:

# Build a simple synthetic image and add shot-like and read noise.
rng = np.random.default_rng(2)
yy, xx = np.mgrid[:128, :128]
clean_image = (
    20
    + 90 * np.exp(-((xx - 42) ** 2 + (yy - 48) ** 2) / (2 * 10**2))
    + 70 * np.exp(-((xx - 88) ** 2 + (yy - 78) ** 2) / (2 * 14**2))
)
shot_noisy = rng.poisson(clean_image).astype(float)
noisy_image = shot_noisy + rng.normal(0, 6, clean_image.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(
    axes,
    [clean_image, shot_noisy, noisy_image],
    ['Clean synthetic image', 'After shot noise', 'After shot + read noise'],
):
    ax.imshow(data)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()



## Dealing with Noise

The PowerPoint makes an important practical distinction:

- If you are fitting a model directly to the image, you may choose to leave the noise in the data and account for it in the model.
- If the noise makes the image hard to interpret or interferes with later analysis, smoothing can help.
- Background and autofluorescence are a different problem from pixel-scale noise and usually need subtraction rather than filtering.



## Mean Filtering

A mean filter replaces each pixel with the average value in a local neighborhood.

- Works well for random pixel-scale noise
- Blurs sharp edges
- Is easy to implement directly as a sliding-window average

The figure below is rebuilt from the original PowerPoint assets for the random-noise, edge, and low-frequency-background examples.

![Mean filter comparison](media/Lecture_2/mean_filter_comparison.png)


In [ ]:

# Brute-force 3x3 mean filter.
def mean_filter_brute_force(image):
    image = image.astype(float)
    H, W = image.shape
    filtered = np.zeros((H - 2, W - 2), dtype=float)
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            window = image[i - 1:i + 2, j - 1:j + 2]
            filtered[i - 1, j - 1] = np.mean(window)
    return filtered


base = np.zeros((80, 80), dtype=float)
base[20:60, 25:55] = 1.0
rng = np.random.default_rng(0)
noisy_square = base + 0.25 * rng.normal(size=base.shape)

brute_mean = mean_filter_brute_force(noisy_square)
library_mean = ndimage.uniform_filter(noisy_square, size=3, mode='nearest')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(noisy_square)
axes[0].set_title('Noisy input')
axes[0].axis('off')
axes[1].imshow(brute_mean)
axes[1].set_title('Brute-force 3x3 mean')
axes[1].axis('off')
axes[2].imshow(library_mean[1:-1, 1:-1])
axes[2].set_title('ndimage.uniform_filter')
axes[2].axis('off')
plt.tight_layout()
plt.show()



## Median Filtering

A median filter replaces the center pixel with the median value in a neighborhood.

- Useful for isolated extreme outliers such as hot pixels or salt-and-pepper noise
- Usually preserves edges better than a mean filter
- Is slower than a mean filter because it requires sorting values in the window

The PowerPoint already contains a clean comparison panel, so we can use it directly.

![Median filter comparison](media/Lecture_2/median_filter_comparison.png)


In [ ]:

# Median filters are especially good for salt-and-pepper noise.
def add_salt_and_pepper(image, amount=0.05, seed=0):
    rng = np.random.default_rng(seed)
    noisy = image.copy()
    n = int(amount * image.size)
    salt_rows = rng.integers(0, image.shape[0], size=n)
    salt_cols = rng.integers(0, image.shape[1], size=n)
    pepper_rows = rng.integers(0, image.shape[0], size=n)
    pepper_cols = rng.integers(0, image.shape[1], size=n)
    noisy[salt_rows, salt_cols] = 1.0
    noisy[pepper_rows, pepper_cols] = 0.0
    return noisy


clean_shape = np.zeros((100, 100), dtype=float)
clean_shape[20:80, 20:80] = 1.0
clean_shape[40:60, 40:60] = 0.3
salt_pepper = add_salt_and_pepper(clean_shape, amount=0.05, seed=1)
mean_filtered = ndimage.uniform_filter(salt_pepper, size=5, mode='nearest')
median_filtered = ndimage.median_filter(salt_pepper, size=5, mode='nearest')

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, data, title in zip(
    axes,
    [clean_shape, salt_pepper, mean_filtered, median_filtered],
    ['Clean image', 'Salt & pepper noise', 'Mean filter (5x5)', 'Median filter (5x5)'],
):
    ax.imshow(data, vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()



## Gaussian Filtering

A Gaussian filter smooths an image by averaging each pixel with its neighbors using weights that are highest at the center and decrease with distance.

- It preserves edges better than a plain mean filter.
- It still blurs fine structure if `sigma` is too large.

In the notation from the slide deck,

$$
I'(x,y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} G(i,j) I(x+i, y+j)
$$

with Gaussian weights

$$
G(i,j) = \frac{1}{2\pi\sigma^2} \exp\left(-\frac{i^2 + j^2}{2\sigma^2}\right).
$$


In [ ]:

# Brute-force 3x3 Gaussian filter.
def gaussian(di, dj, sigma):
    return (1 / (2 * math.pi * sigma**2)) * math.exp(-(di**2 + dj**2) / (2 * sigma**2))


def gaussian_filter_brute_force(image, sigma=1.0):
    image = image.astype(float)
    H, W = image.shape
    filtered = np.zeros((H - 2, W - 2), dtype=float)
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            weighted_sum = 0.0
            total_weight = 0.0
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    weight = gaussian(di, dj, sigma)
                    weighted_sum += weight * image[i + di, j + dj]
                    total_weight += weight
            filtered[i - 1, j - 1] = weighted_sum / total_weight
    return filtered


edge_image = np.zeros((80, 80), dtype=float)
edge_image[:, :40] = 0.2
edge_image[:, 40:] = 1.0
rng = np.random.default_rng(4)
noisy_edge = edge_image + 0.15 * rng.normal(size=edge_image.shape)

brute_gaussian = gaussian_filter_brute_force(noisy_edge, sigma=1.0)
library_gaussian = ndimage.gaussian_filter(noisy_edge, sigma=1.0, mode='nearest')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(noisy_edge, vmin=0, vmax=1)
axes[0].set_title('Noisy edge image')
axes[0].axis('off')
axes[1].imshow(brute_gaussian, vmin=0, vmax=1)
axes[1].set_title('Brute-force Gaussian')
axes[1].axis('off')
axes[2].imshow(library_gaussian[1:-1, 1:-1], vmin=0, vmax=1)
axes[2].set_title('ndimage.gaussian_filter')
axes[2].axis('off')
plt.tight_layout()
plt.show()



## Convolutions in Image Processing

The slide deck introduces filtering as a convolution between an image and a small kernel. For a 3x3 mean filter,

$$
M = \frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

and the filtered image is

$$
I'(x,y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} M(i,j) I(x+i, y+j).
$$

We will use that idea informally here and return to convolutions more explicitly in a later lecture.



## Background Subtraction

Microscopy images often contain unwanted background signal that is not random high-frequency noise.

Common sources include:

1. Autofluorescence or scattered light from the sample
2. Uneven illumination or shading from the optics
3. Non-specific labeling or fluorescence from the medium

If the background is roughly constant, you can subtract a scalar value. If it varies across the field of view, you need to estimate a background image.

The comparison below is rebuilt from the original PowerPoint assets.

![Background subtraction examples](media/Lecture_2/background_subtraction_examples.png)

Another practical approach from the slides is to estimate the background with a very broad blur and subtract it:

```python
background = gaussian_blur(image, large_scale)
image_subtracted = image - background
```


In [ ]:

# Compare constant subtraction with subtraction of a smooth estimated background.
rng = np.random.default_rng(5)
yy, xx = np.mgrid[:160, :160]
background = 30 + 0.18 * xx + 0.12 * yy
signal = (
    80 * np.exp(-((xx - 45) ** 2 + (yy - 55) ** 2) / (2 * 8**2))
    + 110 * np.exp(-((xx - 110) ** 2 + (yy - 100) ** 2) / (2 * 12**2))
)
image_with_background = background + signal + rng.normal(0, 3, background.shape)
constant_background = np.median(image_with_background[:20, :20])
constant_subtracted = np.clip(image_with_background - constant_background, 0, None)
smooth_background = ndimage.gaussian_filter(image_with_background, sigma=20)
background_subtracted = np.clip(image_with_background - smooth_background, 0, None)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, data, title in zip(
    axes,
    [image_with_background, constant_subtracted, smooth_background, background_subtracted],
    ['Original image', 'Subtract constant', 'Estimated background', 'Subtract smooth background'],
):
    ax.imshow(data)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()



## Temporal Median Background Estimation

For a time-lapse movie, the PowerPoint recommends computing the median image across time:

```python
median_background = np.median(stack, axis=0)
corrected = stack - median_background
```

This works best when moving objects do not stay in one place for most of the movie.

![Temporal median examples](media/Lecture_2/temporal_median_examples.png)


In [ ]:

# Build a synthetic movie with one moving bright object on top of a static background.
rng = np.random.default_rng(6)
yy, xx = np.mgrid[:96, :96]
static_background = 25 + 0.10 * xx + 0.05 * yy
stack = []
for t in range(15):
    cx = 15 + 4 * t
    cy = 48 + 10 * np.sin(t / 2)
    moving_spot = 70 * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * 5**2))
    frame = static_background + moving_spot + rng.normal(0, 2, static_background.shape)
    stack.append(frame)
stack = np.stack(stack)
median_background = np.median(stack, axis=0)
corrected = np.clip(stack - median_background, 0, None)
frame_id = 7

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(
    axes,
    [stack[frame_id], median_background, corrected[frame_id]],
    ['Original frame', 'Median background', 'Background-subtracted frame'],
):
    ax.imshow(data)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()



## Key Takeaways

- Bit depth controls how many discrete intensity levels the image can store.
- Dynamic range and contrast are related but not identical.
- Mean, median, and Gaussian filters suppress different kinds of noise and have different tradeoffs.
- Background subtraction targets slowly varying signal rather than pixel-scale noise.
- Temporal medians are a simple and effective way to estimate background in movies with moving objects.
